# Versioning: print versions of AOS-related packages and configs


Owner: **Chris Suberlak** <br>
Last Verified to Run: **2026-05-07** <br>

In [ ]:
# Times Square Parameters
day_obs = 20260507
cscs = "MTAOS,MTHexapod,MTM1M3,MTM2,MTMount,MTRotator,MTDome,MTPtg,MTCamera"

In [ ]:
from lsst.summit.utils import ConsDbClient
import os
import numpy as np

# Only modify no_proxy if it exists (RSP environment)
if "no_proxy" in os.environ:
    os.environ["no_proxy"] += ",.consdb"

consdb_url = "http://consdb-pq.consdb:8080/consdb"
client = ConsDbClient(consdb_url)

In [ ]:
# Times Square Parameters
day_obs = 20260507
cscs = "MTAOS,MTHexapod,MTM1M3,MTM2,MTMount,MTRotator,MTDome,MTPtg,MTCamera"

from lsst.summit.utils import ConsDbClient
import os
import numpy as np

# Only modify no_proxy if it exists (RSP environment)
if "no_proxy" in os.environ:
    os.environ["no_proxy"] += ",.consdb"
consdb_url = "http://consdb-pq.consdb:8080/consdb"
client = ConsDbClient(consdb_url)

# ─────────────────────────────────────────────────────────────────────────────
"""
AOS Software Version Report
============================
Queries EFD for software versions and configurations of all AOS-related CSCs
that were active on a given day_obs. No ts_ofc/ts_wep imports needed.

Fallback strategy
-----------------
  1. Query the 36-hour window centred on day_obs noon  (original window).
  2. If EFD returns no rows (either an empty DataFrame *or* it emits the
     "no data was returned" WARNING), widen to ±7 days and retry.
  3. If still nothing, widen to ±30 days.
  4. The most-recent row from whatever window succeeds is used.
  5. After all CSCs are processed, if every successful hit came from the
     same calendar date (UTC), a single banner is printed instead of
     repeating the timestamp on every row.
"""

import io
import logging
import warnings
import os
import numpy as np
import pandas as pd
from astropy.time import Time, TimeDelta
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient

# Only modify no_proxy if it exists (RSP environment)
if "no_proxy" in os.environ:
    os.environ["no_proxy"] += ",.consdb"

efd_client = makeEfdClient()

# ── silence the noisy EFD "no data" warning for the whole cell ───────────────
# The warning fires both through the logging system and warnings.warn;
# suppress both so the fallback logic can handle the empty result quietly.
logging.getLogger("lsst.summit.utils.efdUtils").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore",
    message=".*no data was returned.*",
    category=UserWarning,
)

# ── helpers ──────────────────────────────────────────────────────────────────

def _safe_getEfdData(efd_client, topic, begin, end):
    """Call getEfdData, returning an empty DataFrame on 'no data' or any error."""
    try:
        df = getEfdData(efd_client, topic, begin=begin, end=end)
        return df if df is not None else pd.DataFrame()
    except Exception:
        return pd.DataFrame()


def _query_with_fallback(efd_client, topic, t_start, t_end):
    """
    Try progressively wider time windows until rows are returned.

    Returns
    -------
    df            : pd.DataFrame (possibly empty if nothing found after 30 days)
    window_label  : str describing which window succeeded
    """
    windows = [
        (t_start, t_end, "requested window"),
        (t_start - TimeDelta(7 * 86400, format='sec'),
         t_end   + TimeDelta(7 * 86400, format='sec'), "±7 d fallback"),
        (t_start - TimeDelta(30 * 86400, format='sec'),
         t_end   + TimeDelta(30 * 86400, format='sec'), "±30 d fallback"),
    ]
    for begin, end, label in windows:
        df = _safe_getEfdData(efd_client, topic, begin, end)
        if len(df) > 0:
            return df, label
    return pd.DataFrame(), "no data found"


def _fmt_ts(ts):
    """Format a pandas Timestamp or astropy Time as 'YYYY-MM-DD HH:MM UTC'."""
    if isinstance(ts, pd.Timestamp):
        base = ts.tz_localize(None) if ts.tzinfo else ts
        return base.strftime("%Y-%m-%d %H:%M UTC")
    return Time(ts, scale='utc').strftime("%Y-%m-%d %H:%M UTC")


def _utc_date(ts):
    """Return 'YYYY-MM-DD' for a pandas Timestamp or astropy Time."""
    if isinstance(ts, pd.Timestamp):
        base = ts.tz_localize(None) if ts.tzinfo else ts
        return base.strftime("%Y-%m-%d")
    return Time(ts, scale='utc').to_value('iso', subfmt='date')


def _ts_to_pandas(ts):
    """Normalise any timestamp to a tz-naive pandas Timestamp for comparison."""
    if isinstance(ts, pd.Timestamp):
        return ts.tz_localize(None) if ts.tzinfo else ts
    return pd.Timestamp(Time(ts, scale='utc').unix, unit='s')


# ── time window ──────────────────────────────────────────────────────────────

day_str  = str(day_obs)
noon_iso = f"{day_str[:4]}-{day_str[4:6]}-{day_str[6:8]}T12:00:00"
t_noon   = Time(noon_iso, scale='utc')
t_start  = t_noon - TimeDelta(12 * 3600, format='sec')
t_end    = t_noon + TimeDelta(24 * 3600, format='sec')

print(f"AOS Software Version Report — dayObs {day_obs}")
print(f"Requested window: {t_start.iso}  →  {t_end.iso}")
print("=" * 100)

# ── main loop ────────────────────────────────────────────────────────────────

cscs_to_check = cscs.split(',')

print("\n1. SOFTWARE VERSIONS")
print("-" * 100)
print(f"{'CSC':<16s} {'CSC Version':<18s} {'XML Version':<14s} {'Most recent timestamp':<24s} {'Subsystem Versions'}")
print("-" * 100)

version_data = []

for csc in cscs_to_check:
    topic = f"lsst.sal.{csc}.logevent_softwareVersions"
    try:
        df, window_label = _query_with_fallback(efd_client, topic, t_start, t_end)

        if len(df) > 0:
            row     = df.iloc[-1]
            ts      = df.index[-1]
            csc_ver = str(row.get('cscVersion',       '?'))
            xml_ver = str(row.get('xmlVersion',        '?'))
            sub_ver = str(row.get('subsystemVersions', ''))
            sal_ver = str(row.get('salVersion',        ''))

            ts_str      = _fmt_ts(ts)
            sub_display = sub_ver[:38] + '…' if len(sub_ver) > 38 else sub_ver

            print(f"{csc:<16s} {csc_ver:<18s} {xml_ver:<14s} {ts_str:<24s} {sub_display}")

            version_data.append(dict(
                csc=csc,
                csc_version=csc_ver,
                xml_version=xml_ver,
                subsystem_versions=sub_ver,
                sal_version=sal_ver,
                timestamp=ts,
                window=window_label,
            ))
        else:
            print(f"{csc:<16s} {'(no data in any window)':}")

    except Exception as e:
        print(f"{csc:<16s} ERROR — {str(e)[:70]}")

# ── MTAOS subsystem breakdown ────────────────────────────────────────────────

print("\n  MTAOS subsystem breakdown:")
mtaos_versions = [v for v in version_data if v['csc'] == 'MTAOS']
if mtaos_versions:
    for part in mtaos_versions[0]['subsystem_versions'].split(','):
        part = part.strip()
        if part:
            print(f"    {part}")
    print(f"  (reported at {_fmt_ts(mtaos_versions[0]['timestamp'])})")
else:
    print("    (no MTAOS data)")

# ── same-day summary banner ───────────────────────────────────────────────────

print()
if version_data:
    timestamps     = [v['timestamp'] for v in version_data]
    dates          = [_utc_date(ts) for ts in timestamps]
    unique_dates   = set(dates)
    requested_date = f"{day_str[:4]}-{day_str[4:6]}-{day_str[6:8]}"

    if len(unique_dates) == 1:
        common_date = next(iter(unique_dates))
        latest_ts   = max(timestamps, key=_ts_to_pandas)

        print("=" * 100)
        print(f"ℹ️  All CSCs reported version information from the same date.")
        print(f"   Most recent versioning information is from: {_fmt_ts(latest_ts)}")
        if common_date != requested_date:
            print(
                f"   ⚠️  Note: this differs from the requested day_obs date "
                f"({requested_date}); a fallback window was used for all CSCs."
            )
        print("=" * 100)

    else:
        # Mixed dates — show per-CSC detail
        print("=" * 100)
        print("ℹ️  CSCs reported from different dates — per-CSC timestamps:")
        for v in version_data:
            flag = f"  [{v['window']}]" if v['window'] != "requested window" else ""
            print(f"   {v['csc']:<16s} {_fmt_ts(v['timestamp'])}{flag}")
        print("=" * 100)

In [ ]:
t_start

In [ ]:
# =============================================================================
# 3. Configurations Available (what overrides exist)
# =============================================================================

avail_data = []

for csc in ['MTAOS', 'MTHexapod', 'MTM1M3', 'MTM2']:
    topic = f"lsst.sal.{csc}.logevent_configurationsAvailable"
    try:
        df, window_label = _query_with_fallback(efd_client, topic, t_start, t_end)

        if len(df) > 0:
            row       = df.iloc[-1]
            ts        = df.index[-1]
            overrides = str(row.get('overrides', ''))
            avail_data.append(dict(
                csc=csc,
                overrides=[ov.strip() for ov in overrides.split(',') if ov.strip()],
                timestamp=ts,
                window=window_label,
            ))
        else:
            avail_data.append(dict(csc=csc, overrides=None, timestamp=None, window="no data found"))

    except Exception as e:
        avail_data.append(dict(csc=csc, overrides=None, timestamp=None, window=f"ERROR: {str(e)[:60]}"))

# ── print ─────────────────────────────────────────────────────────────────────

print("\n3. AVAILABLE CONFIGURATION OVERRIDES")
print("-" * 90)

for a in avail_data:
    if a['timestamp'] is not None:
        ts_str = _fmt_ts(a['timestamp'])
        print(f"  {a['csc']}:  (as of {ts_str})")
        if a['overrides']:
            for ov in a['overrides']:
                print(f"    - {ov}")
        else:
            print("    (none)")
    elif a['window'].startswith("ERROR"):
        print(f"  {a['csc']}: ERROR — {a['window'][7:]}")
    else:
        print(f"  {a['csc']}: (no data in any window)")

# ── same-day summary banner ───────────────────────────────────────────────────

successful_avail = [a for a in avail_data if a['timestamp'] is not None]
print()
if successful_avail:
    timestamps     = [a['timestamp'] for a in successful_avail]
    unique_dates   = set(_utc_date(ts) for ts in timestamps)
    requested_date = f"{day_str[:4]}-{day_str[4:6]}-{day_str[6:8]}"

    if len(unique_dates) == 1:
        common_date = next(iter(unique_dates))
        latest_ts   = max(timestamps, key=_ts_to_pandas)
        print("=" * 90)
        print(f"ℹ️  All CSCs reported available configurations from the same date.")
        print(f"   Most recent information is from: {_fmt_ts(latest_ts)}")
        if common_date != requested_date:
            print(f"   ⚠️  Note: this differs from the requested day_obs date ({requested_date}); a fallback window was used.")
        print("=" * 90)
    else:
        print("=" * 90)
        print("ℹ️  CSCs reported from different dates — per-CSC timestamps:")
        for a in successful_avail:
            flag = f"  [{a['window']}]" if a['window'] != "requested window" else ""
            print(f"   {a['csc']:<16s} {_fmt_ts(a['timestamp'])}{flag}")
        print("=" * 90)


# =============================================================================
# 4. CSC Summary States (was it running that night?)
# =============================================================================

state_names = {1: 'DISABLED', 2: 'ENABLED', 3: 'FAULT', 4: 'OFFLINE', 5: 'STANDBY'}

state_data = []

for csc in cscs_to_check:
    topic = f"lsst.sal.{csc}.logevent_summaryState"
    try:
        df, window_label = _query_with_fallback(efd_client, topic, t_start, t_end)

        if len(df) > 0:
            states       = df['summaryState'].values
            state_labels = [state_names.get(int(s), f'UNKNOWN({int(s)})') for s in states]
            unique_seq   = list(dict.fromkeys(state_labels))   # ordered, deduplicated
            n_faults     = sum(1 for s in states if int(s) == 3)
            ts           = df.index[-1]

            state_data.append(dict(
                csc=csc,
                final_state=state_labels[-1],
                transitions=unique_seq,
                n_faults=n_faults,
                timestamp=ts,
                window=window_label,
            ))
        else:
            state_data.append(dict(
                csc=csc, final_state=None, transitions=None,
                n_faults=0, timestamp=None, window="no data found",
            ))

    except Exception as e:
        state_data.append(dict(
            csc=csc, final_state=None, transitions=None,
            n_faults=0, timestamp=None, window=f"ERROR: {str(e)[:60]}",
        ))

# ── print ─────────────────────────────────────────────────────────────────────

print("\n4. CSC SUMMARY STATES DURING NIGHT")
print("-" * 90)

for s in state_data:
    if s['timestamp'] is not None:
        fault_str   = f" ({s['n_faults']} FAULT events)" if s['n_faults'] > 0 else ""
        transitions = " → ".join(s['transitions'])
        ts_str      = _fmt_ts(s['timestamp'])
        print(f"  {s['csc']:<16s} final={s['final_state']:<10s} transitions: {transitions}{fault_str}  (last event: {ts_str})")
    elif s['window'].startswith("ERROR"):
        print(f"  {s['csc']:<16s} ERROR — {s['window'][7:]}")
    else:
        print(f"  {s['csc']:<16s} (no state events in any window)")

# ── same-day summary banner ───────────────────────────────────────────────────

successful_states = [s for s in state_data if s['timestamp'] is not None]
print()
if successful_states:
    timestamps     = [s['timestamp'] for s in successful_states]
    unique_dates   = set(_utc_date(ts) for ts in timestamps)
    requested_date = f"{day_str[:4]}-{day_str[4:6]}-{day_str[6:8]}"

    if len(unique_dates) == 1:
        common_date = next(iter(unique_dates))
        latest_ts   = max(timestamps, key=_ts_to_pandas)
        print("=" * 90)
        print(f"ℹ️  All CSCs reported state information from the same date.")
        print(f"   Most recent state information is from: {_fmt_ts(latest_ts)}")
        if common_date != requested_date:
            print(f"   ⚠️  Note: this differs from the requested day_obs date ({requested_date}); a fallback window was used.")
        print("=" * 90)
    else:
        print("=" * 90)
        print("ℹ️  CSCs reported from different dates — per-CSC timestamps:")
        for s in successful_states:
            flag = f"  [{s['window']}]" if s['window'] != "requested window" else ""
            print(f"   {s['csc']:<16s} {_fmt_ts(s['timestamp'])}{flag}")
        print("=" * 90)


# =============================================================================
# 5. MTAOS-specific: OFC controller config and key parameters
# =============================================================================

mtaos_log_data = dict(messages=[], timestamp_first=None, timestamp_last=None, error=None)

try:
    df, window_label = _query_with_fallback(
        efd_client, "lsst.sal.MTAOS.logevent_logMessage", t_start, t_end
    )

    if len(df) > 0:
        config_keywords = [
            'config', 'controller', 'gain', 'ofc', 'vmode', 'truncat',
            'normali', 'sensor', 'intrinsic', 'comp_dof', 'bending',
            'donut', 'radius', 'defocus', 'stress', 'pipeline',
        ]
        matched = [
            (ts, str(row.get('message', '')))
            for ts, row in df.iterrows()
            if any(kw in str(row.get('message', '')).lower() for kw in config_keywords)
        ]
        mtaos_log_data['messages']        = matched
        mtaos_log_data['total_log_count'] = len(df)
        mtaos_log_data['timestamp_first'] = df.index[0]
        mtaos_log_data['timestamp_last']  = df.index[-1]
        mtaos_log_data['window']          = window_label
    else:
        mtaos_log_data['error'] = "no log messages found in any window"

except Exception as e:
    mtaos_log_data['error'] = str(e)

# ── print ─────────────────────────────────────────────────────────────────────

print("\n5. MTAOS RUNTIME PARAMETERS (from log messages)")
print("-" * 90)

if mtaos_log_data['error']:
    print(f"  ERROR: {mtaos_log_data['error']}")
else:
    total   = mtaos_log_data['total_log_count']
    matched = mtaos_log_data['messages']
    t_first = _fmt_ts(mtaos_log_data['timestamp_first'])
    t_last  = _fmt_ts(mtaos_log_data['timestamp_last'])

    print(f"  Found {total} MTAOS log messages ({t_first} → {t_last}), filtering for config keywords...")

    if matched:
        for ts, msg in matched[:30]:
            print(f"    [{_fmt_ts(ts)}] {msg[:120]}")
        if len(matched) > 30:
            print(f"    ... ({len(matched) - 30} more matching messages)")
    else:
        print("  No config-related log messages found")

In [ ]:
# =============================================================================
# 6. Summit config file paths (if accessible)
# =============================================================================
print("\n6. SUMMIT CONFIGURATION FILES")
print("-" * 90)

import os, yaml

config_base = "/net/obs-env/auto_base_packages/ts_config_mttcs/MTAOS"

# Find the schema version used from configurationApplied
mtaos_config = [c for c in config_data if c['csc'] == 'MTAOS']
if mtaos_config:
    schema = mtaos_config[0]['schema_version']
    config_file = mtaos_config[0]['configurations']
    config_dir = f"{config_base}/{schema}"
    print(f"  Schema version: {schema}")
    print(f"  Config file: {config_file}")
    print(f"  Config dir: {config_dir}")

    init_path = f"{config_dir}/{config_file}"
    if os.path.isfile(init_path):
        print(f"\n  Contents of {config_file}:")
        with open(init_path) as f:
            config = yaml.safe_load(f)
        # Show key AOS parameters
        key_params = [
            'control_vmodes', 'subtract_intrinsics', 'used_dofs',
            'stress_scale_approach', 'stress_scale_factor',
            'm1m3_stress_limit', 'm2_stress_limit',
            'dz_threshold_min', 'dz_threshold_max',
            'raise_on_large_defocus', 'zernike_table_name',
            'zernike_column_pattern', 'num_expected_tables',
            'pipeline_n_processes',
        ]
        for k in key_params:
            if k in config:
                val = config[k]
                if isinstance(val, list) and len(val) > 10:
                    val = f"[{val[0]}, ..., {val[-1]}] ({len(val)} elements)"
                print(f"    {k}: {val}")

        # Show OFC controller config
        ofc_config = config.get('closed_loop_ofc_configuration', {})
        if ofc_config:
            print(f"\n  OFC configuration overrides:")
            for k, v in ofc_config.items():
                print(f"    {k}: {v}")

        # Read the controller init.yaml
        ofc_init = f"{config_base}/ofc/configurations/init.yaml"
        if os.path.isfile(ofc_init):
            with open(ofc_init) as f:
                ctrl = yaml.safe_load(f)
            print(f"\n  OFC controller (init.yaml):")
            for k in ['name', 'xref', 'truncation_index',
                       'normalization_weights_filename', 'rotation_offset']:
                if k in ctrl:
                    print(f"    {k}: {ctrl[k]}")
            # Gains
            kp = ctrl.get('kp', [])
            if isinstance(kp, list):
                kp_hex = kp[:10] if len(kp) >= 10 else kp
                kp_bend = kp[10:12] if len(kp) >= 12 else []
                print(f"    kp (hex): {kp_hex[0]} (all 10)" if len(set(kp_hex)) == 1
                      else f"    kp (hex): {kp_hex}")
                print(f"    kp (bend): {kp_bend[0]} (all 40)" if kp_bend and len(set(kp[10:])) == 1
                      else f"    kp (bend): {kp_bend}")
            ki = ctrl.get('ki', '?')
            kd = ctrl.get('kd', '?')
            print(f"    ki: {ki}")
            print(f"    kd: {kd}")
    else:
        print(f"\n  Config file not found at {init_path}")
        print(f"  (expected — this notebook may be running at USDF, not summit)")
else:
    print("  No MTAOS configurationApplied data found")

In [ ]:
# =============================================================================
# 7. Summary table
# =============================================================================
print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)

summary_rows = []
for v in version_data:
    c = [x for x in config_data if x['csc'] == v['csc']]
    summary_rows.append({
        'CSC': v['csc'],
        'Version': v['csc_version'],
        'XML': v['xml_version'],
        'Config': c[0]['configurations'] if c else '?',
        'Schema': c[0]['schema_version'] if c else '?',
        'Last Started': str(v['timestamp'])[:19],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# MTAOS subsystem versions as separate table
print(f"\nMTAOS Subsystem Packages:")
if mtaos_versions:
    sub_str = mtaos_versions[0]['subsystem_versions']
    for part in sub_str.split(','):
        k, _, v = part.strip().partition('=')
        if k:
            print(f"  {k:<20s} {v}")